# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# Configuration: secrets stay in .env (never commit this file)
import os
from dotenv import load_dotenv
load_dotenv('.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise RuntimeError('Configuration | OPENAI_API_KEY is missing. Check the .env file in this folder.')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
TEMPERATURE = 0.7
EMAIL_SENDER = os.getenv('EMAIL_SENDER', '')
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD', '')
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ['salt', 'pepper', 'oil', 'garlic', 'water']
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


In [3]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [6]:
import json
import time
import re
import openai
from pathlib import Path
from pydantic import ValidationError


def parse_json_response(raw_text: str) -> dict:
    """Parse the model response independently from the API call."""
    if not isinstance(raw_text, str):
        raise TypeError('Model response content is not a string')
    text = raw_text.strip()
    match = re.fullmatch(r'```(?:json)?\s*\n?(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if match:
        text = match.group(1).strip()
    return json.loads(text)


def validate_meal_plan(data: dict) -> dict:
    """Validate against the Pydantic schema; return a plain dictionary."""
    return MealPlan.model_validate(data).model_dump()


def load_json_file(filename: str) -> dict:
    """Load a JSON fixture. Kept separate to make file/JSON errors testable."""
    path = Path(filename)
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except FileNotFoundError as exc:
        raise FileNotFoundError(f'load_json_file | FileNotFoundError: {path} not found. Check the file path.') from exc
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'load_json_file | JSONDecodeError: {path}, line {exc.lineno}, '
            f'column {exc.colno}: {exc.msg}. Check JSON syntax.'
        ) from exc


def error_result(stage: str, exc: Exception, suggestion: str, start_time: float) -> dict:
    """Keep the same keys on both success and failure."""
    details = str(exc)
    if isinstance(exc, json.JSONDecodeError):
        details += f' (line {exc.lineno}, column {exc.colno})'
    if isinstance(exc, ValidationError):
        details = '; '.join(
            f"{'.'.join(map(str, item['loc']))}: {item['msg']}" for item in exc.errors()
        )
    status_code = getattr(exc, 'status_code', None)
    if status_code:
        details += f' (HTTP {status_code})'
    return {
        'success': False,
        'meal_plan': None,
        'generation_time': round(time.time() - start_time, 2),
        'error': f'{stage} | {type(exc).__name__}: {details}. Suggestion: {suggestion}'
    }


class MealPlanGenerator:
    """Generate a meal plan; each part is isolated for testing."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)

    def build_inputs(self, dietary_preference, theme_requests, additional_preferences):
        """Prepare the prompt inputs only."""
        return {
            'dietary_preference': dietary_preference,
            'theme_requests': theme_requests.strip() if theme_requests and theme_requests.strip() else 'random diverse cuisines',
            'additional_preferences': additional_preferences.strip() if additional_preferences and additional_preferences.strip() else 'none',
            'format_instructions': self.recipe_parser.get_format_instructions(),
        }

    def request_plan(self, inputs):
        """Call the model only; no JSON parsing here."""
        chain = MEAL_PLAN_PROMPT | self.llm
        return chain.invoke(inputs)

    def generate(self, dietary_preference: str, theme_requests: str = 'random', additional_preferences: str = 'none') -> dict:
        start_time = time.time()
        try:
            inputs = self.build_inputs(dietary_preference, theme_requests, additional_preferences)
            result = self.request_plan(inputs)
        except openai.APIError as exc:
            return error_result('generate.request_plan', exc, 'Check API access, status and network; retry only if transient.', start_time)
        except (OSError, ConnectionError, TimeoutError) as exc:
            return error_result('generate.request_plan', exc, 'Check your connection and retry.', start_time)

        try:
            data = parse_json_response(result.content)
        except (json.JSONDecodeError, TypeError) as exc:
            return error_result('generate.parse_json_response', exc, 'Check the model JSON response.', start_time)

        try:
            meal_plan = validate_meal_plan(data)
        except ValidationError as exc:
            return error_result('generate.validate_meal_plan', exc, 'Check recipe count, themes and ingredients.', start_time)

        return {
            'success': True,
            'meal_plan': meal_plan,
            'generation_time': round(time.time() - start_time, 2),
            'error': None,
        }


# Initialize generator
generator = MealPlanGenerator()


## 6. Email Functionality

In [7]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_ingredients(recipe: dict) -> str:
    """Shared ingredient formatter for email and Gradio."""
    return ', '.join(f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients'])


def format_meal_plan_email(meal_plan: dict) -> str:
    lines = ['=' * 40, f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})", '=' * 40, '',
             'SHOPPING LIST', '-' * 20]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.extend(['', '=' * 40, 'RECIPES', '=' * 40])
    for recipe in meal_plan['recipes']:
        lines.extend([f"\n{recipe['day'].upper()} - {recipe['theme']}", recipe['dish_name'],
                      f"Ingredients: {format_ingredients(recipe)}", f"Instructions: {recipe['instructions']}"])
    return '\n'.join(lines)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send the already-generated plan (do not generate another one)."""
    if not EMAIL_SENDER or not EMAIL_PASSWORD:
        return {'success': False, 'message': 'send_meal_plan_email | ConfigurationError: Email credentials missing. Set EMAIL_SENDER and EMAIL_PASSWORD in .env.'}
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = recipient_email
        msg['Subject'] = 'Your Weekly Meal Plan'
        msg.attach(MIMEText(format_meal_plan_email(meal_plan), 'plain'))
        with smtplib.SMTP('smtp.gmail.com', 587, timeout=15) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)
        return {'success': True, 'message': f'Email sent to {recipient_email}'}
    except (smtplib.SMTPException, OSError) as exc:
        return {'success': False, 'message': f'send_meal_plan_email | {type(exc).__name__}: {exc}. Check email settings/network.'}


## 7. Validation

In [8]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Quick independent checks on the output (in addition to Pydantic)."""
    if not meal_plan_result.get('success'):
        return {'passed': False, 'errors': [meal_plan_result.get('error', 'Unknown failure')]}
    meal_plan = meal_plan_result['meal_plan']
    errors = []
    if len(meal_plan['recipes']) != len(DAYS):
        errors.append(f"Expected {len(DAYS)} recipes, got {len(meal_plan['recipes'])}")
    themes = [r['theme'].lower() for r in meal_plan['recipes']]
    if len(set(themes)) != len(themes):
        errors.append('Duplicate themes found')
    for recipe in meal_plan['recipes']:
        if len(recipe['ingredients']) != INGREDIENTS_PER_RECIPE:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients; expected {INGREDIENTS_PER_RECIPE}")
    return {'passed': len(errors) == 0, 'errors': errors}


## 8. Gradio Interface

In [9]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    lines = ['## Shopping List\n']
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return '\n'.join(lines)


def format_single_recipe(recipe: dict) -> str:
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{format_ingredients(recipe)}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Yield both display output and the saved plan state."""
    yield 'Generating your meal plan...', None
    result = generator.generate(dietary_pref, theme_requests, additional_prefs)
    if not result['success']:
        yield f"Error: {result['error']}", None
        return
    meal_plan = result['meal_plan']
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + '\n\n---\n\n## Recipes\n\n'
    yield output, meal_plan
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + '\n'
        yield output, meal_plan
        pytime.sleep(0.05)
    validation = run_basic_validation(result)
    status = 'All checks passed' if validation['passed'] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output, meal_plan


def send_email_ui(email, saved_plan):
    """Email the same plan shown on screen; no second OpenAI call."""
    if not email or '@' not in email:
        return 'Enter a valid email address.'
    if not saved_plan:
        return 'Please generate a meal plan first.'
    return send_meal_plan_email(email, saved_plan)['message']


with gr.Blocks(title='Weekly Meal Planner') as app:
    gr.Markdown('# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.')
    current_plan = gr.State(value=None)
    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(choices=['omnivore', 'vegetarian', 'vegan', 'keto'], value='omnivore', label='Diet')
            theme_input = gr.Textbox(label='Themes (optional)', placeholder='Italian, Mexican...', lines=1)
            additional_input = gr.Textbox(label='Preferences (optional)', placeholder='no nuts, quick meals...', lines=1)
            generate_btn = gr.Button('Generate', variant='primary')
            gr.Markdown('---')
            email_input = gr.Textbox(label='Email', placeholder='you@example.com')
            send_btn = gr.Button('Email Plan')
            email_status = gr.Textbox(label='Status', interactive=False)
        with gr.Column(scale=2):
            output_display = gr.Markdown('Click **Generate** to start.')
    generate_btn.click(fn=generate_plan_streaming,
                       inputs=[dietary_dropdown, theme_input, additional_input],
                       outputs=[output_display, current_plan])
    send_btn.click(fn=send_email_ui, inputs=[email_input, current_plan], outputs=[email_status])


c:\Users\Roodabeh Seif\miniconda3-pbe\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 9. Launch Application

In [ ]:
# Local-only UI; do not publish a public link during the lab.
# Optional for manual testing after all cells work:
# app.launch(share=False)


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [ ]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

## 11. Refactoring tests — offline (no API charges)
These tests intentionally provide good and broken data. They should not call OpenAI. Capture your own output as evidence.


In [10]:
# Test fixtures and error checks — no OpenAI calls
import tempfile
from pathlib import Path
from unittest.mock import patch
from types import SimpleNamespace

sample = {
    'dietary_preference': 'vegetarian',
    'recipes': [
        {'day': day, 'theme': theme, 'dish_name': f'{theme} dinner',
         'ingredients': [{'name': f'Ingredient {j}', 'quantity': '1 unit'} for j in range(5)],
         'instructions': 'Cook and serve.'}
        for day, theme in zip(DAYS, ['Italian', 'Mexican', 'Japanese', 'Thai', 'Greek', 'French', 'Indian'])
    ],
    'shopping_list': [{'name': 'Ingredient 0', 'total_quantity': '7 units', 'used_in': ['example dinner']}]
}
assert len(validate_meal_plan(parse_json_response(json.dumps(sample)))['recipes']) == 7
print('PASS valid JSON + Pydantic schema')

try:
    parse_json_response('{"recipes": [}')
except json.JSONDecodeError as exc:
    print(f'PASS invalid JSON: {type(exc).__name__}, line {exc.lineno}, column {exc.colno}')

try:
    validate_meal_plan({**sample, 'recipes': sample['recipes'][:6]})
except ValidationError as exc:
    print('PASS invalid data:', exc.errors()[0]['loc'], exc.errors()[0]['msg'])

try:
    load_json_file('missing_file_for_lab.json')
except FileNotFoundError as exc:
    print('PASS missing file:', exc)

with tempfile.TemporaryDirectory() as tmp:
    invalid = Path(tmp) / 'malformed.json'
    invalid.write_text('{wrong json', encoding='utf-8')
    try:
        load_json_file(str(invalid))
    except ValueError as exc:
        print('PASS malformed file:', exc)

# Simulate an API error without sending a request.
with patch.object(generator, 'request_plan', side_effect=openai.APIError('Simulated API failure', request=None, body=None)):
    test_result = generator.generate('vegetarian')
    assert not test_result['success'] and 'generate.request_plan' in test_result['error']
    print('PASS simulated API error:', test_result['error'])


PASS valid JSON + Pydantic schema
PASS invalid JSON: JSONDecodeError, line 1, column 14
PASS invalid data: ('recipes',) Value error, Meal plan must have exactly 7 recipes, got 6
PASS missing file: load_json_file | FileNotFoundError: missing_file_for_lab.json not found. Check the file path.
PASS malformed file: load_json_file | JSONDecodeError: C:\Users\ROODAB~1\AppData\Local\Temp\tmp3fkb7moe\malformed.json, line 1, column 2: Expecting property name enclosed in double quotes. Check JSON syntax.
PASS simulated API error: generate.request_plan | APIError: Simulated API failure. Suggestion: Check API access, status and network; retry only if transient.


## 12. Optional real API verification (costs API credits)
Only run once after you have checked the refactoring. This is your actual post-refactoring baseline comparison. Do not claim the offline test above was a live API run.


In [11]:
# Run once with your own credentials; takes real API credits
refactored_result = generator.generate('vegetarian', 'Italian, Mexican', 'no mushrooms')
print('Success:', refactored_result['success'])
if refactored_result['success']:
    print('Recipes:', len(refactored_result['meal_plan']['recipes']))
    print('Validation:', run_basic_validation(refactored_result))
else:
    print('Error:', refactored_result['error'])


Success: True
Recipes: 7
Validation: {'passed': True, 'errors': []}
